# Lab Assignment - 6
**Course:** CSET343 - AI in Healthcare  
**Topic:** Automated Diagnosis of Lung Disease (Pneumonia vs. Normal) using Chest X-Ray Images  

---
### Objective:
Design, implement, and evaluate machine learning pipelines for automated detection of pneumonia from Chest X-Ray (CXR) scans. This notebook implements:
1. Data Acquisition and Exploration (Class balance, visualization).
2. Handcrafted Feature Extraction: **HOG** (shape/edges), **LBP** (texture), and **GLCM** (spatial gray-level relationship).
3. Machine Learning Baselines: **Support Vector Machine (SVM)** and **Random Forest (RF)** with hyperparameter optimization.
4. Deep Learning Classifier: A custom **Convolutional Neural Network (CNN)** built with PyTorch.
5. Evaluation: **Accuracy, Precision, Recall, F1-Score, AUROC**, Confusion Matrices, ROC Curves, and Clinical Misclassification Analysis.

---
## Task 1: Data Acquisition and Exploration

In [ ]:
import os
import glob
import shutil
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import kagglehub

# 1. Download and set up dataset
target_dir = os.path.abspath("datasets/chest_xray")
os.makedirs("datasets", exist_ok=True)

if not os.path.exists(target_dir):
    print("Downloading Chest X-Ray dataset via KaggleHub...")
    raw_path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
    src = raw_path
    if os.path.exists(os.path.join(raw_path, "chest_xray")):
        src = os.path.join(raw_path, "chest_xray")
    elif os.path.exists(os.path.join(raw_path, "chest_xray", "chest_xray")):
        src = os.path.join(raw_path, "chest_xray", "chest_xray")
    shutil.copytree(src, target_dir)
    print("Dataset download and setup complete!")
else:
    print(f"Dataset ready at: {target_dir}")

In [ ]:
# 2. Class distribution analysis
normal_files = sorted(glob.glob(os.path.join(target_dir, "**", "NORMAL", "*.jpeg"), recursive=True) + \
                      glob.glob(os.path.join(target_dir, "**", "NORMAL", "*.png"), recursive=True))
pneumonia_files = sorted(glob.glob(os.path.join(target_dir, "**", "PNEUMONIA", "*.jpeg"), recursive=True) + \
                         glob.glob(os.path.join(target_dir, "**", "PNEUMONIA", "*.png"), recursive=True))

total_images = len(normal_files) + len(pneumonia_files)
print("=== Class Distribution & Balance ===")
print(f"Total Images: {total_images}")
print(f"Normal Scans:    {len(normal_files)} ({len(normal_files)/total_images*100:.2f}%)")
print(f"Pneumonia Scans: {len(pneumonia_files)} ({len(pneumonia_files)/total_images*100:.2f}%)")

> **Observations on Data Balance:**  
> The dataset presents a significant class imbalance (~73% Pneumonia vs ~27% Normal). This reflects real-world clinical datasets where positive diagnostic cases are overrepresented in referral center databases. To address this, model evaluation must emphasize **Precision, Recall (Sensitivity), F1-Score, and AUROC** rather than relying solely on overall accuracy.

In [ ]:
# 3. Display sample scans from each class
np.random.seed(42)
sample_normal = np.random.choice(normal_files, 5, replace=False)
sample_pneumonia = np.random.choice(pneumonia_files, 5, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    img_n = Image.open(sample_normal[i]).convert('L').resize((128, 128))
    img_p = Image.open(sample_pneumonia[i]).convert('L').resize((128, 128))
    
    axes[0, i].imshow(img_n, cmap='gray')
    axes[0, i].set_title(f"Normal #{i+1}", fontsize=10, color='darkgreen')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(img_p, cmap='gray')
    axes[1, i].set_title(f"Pneumonia #{i+1}", fontsize=10, color='darkred')
    axes[1, i].axis('off')

plt.suptitle("Sample Chest X-Rays: Normal (Top) vs. Pneumonia (Bottom)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Task 2: Data Preprocessing and Feature Extraction

In Task 2, we implement feature extraction pipeline:
1. Resize images to `128x128` and normalize pixel values to `[0, 1]`.
2. Extract three complementary feature descriptors:
   - **HOG (Histogram of Oriented Gradients)**: Captures macroscopic anatomical shapes and rib cage structure.
   - **LBP (Local Binary Patterns)**: Captures micro-texture variations in lung parenchyma.
   - **GLCM (Gray-Level Co-occurrence Matrix)**: Captures second-order spatial gray-level properties (contrast, dissimilarity, homogeneity, energy, correlation, ASM).
3. Concatenate feature vectors and partition data into **70% Training, 15% Validation, and 15% Test** splits.

In [ ]:
import time
from multiprocessing import Pool, cpu_count
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def extract_hog_features(img_gray):
    return hog(img_gray, orientations=9, pixels_per_cell=(16, 16),
               cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False).astype(np.float32)

def extract_lbp_features(img_gray):
    img_u8 = (img_gray * 255).astype(np.uint8)
    radius = 1
    n_points = 8 * radius
    lbp = local_binary_pattern(img_u8, n_points, radius, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, n_points + 3), range=(0, n_points + 2))
    hist = hist.astype(np.float32)
    return hist / (hist.sum() + 1e-6)

def extract_glcm_features(img_gray):
    img_u8 = (img_gray * 255).astype(np.uint8)
    img_q = (img_u8 // 4).astype(np.uint8) # 64 gray levels
    glcm = graycomatrix(img_q, distances=[1, 3], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=64, symmetric=True, normed=True)
    props = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']
    feats = []
    for prop in props:
        feats.extend(graycoprops(glcm, prop).ravel())
    return np.array(feats, dtype=np.float32)

def extract_single_file_features(img_path):
    img = Image.open(img_path).convert('L').resize((128, 128))
    img_arr = np.array(img, dtype=np.float32) / 255.0
    
    h_feat = extract_hog_features(img_arr)
    l_feat = extract_lbp_features(img_arr)
    g_feat = extract_glcm_features(img_arr)
    return np.concatenate([h_feat, l_feat, g_feat])

print("Feature extraction functions ready.")

In [ ]:
# 4. Stratified Data Splitting & Feature Extraction
all_files = normal_files + pneumonia_files
all_labels = [0] * len(normal_files) + [1] * len(pneumonia_files)

files_train_val, files_test, labels_train_val, labels_test = train_test_split(
    all_files, all_labels, test_size=0.15, random_state=42, stratify=all_labels
)
files_train, files_val, labels_train, labels_val = train_test_split(
    files_train_val, labels_train_val, test_size=0.17647, random_state=42, stratify=labels_train_val
)

print(f"Train set: {len(files_train)} images ({len(files_train)/len(all_files)*100:.1f}%)")
print(f"Val set:   {len(files_val)} images ({len(files_val)/len(all_files)*100:.1f}%)")
print(f"Test set:  {len(files_test)} images ({len(files_test)/len(all_files)*100:.1f}%)")

# Extract features in parallel
print(f"\nExtracting feature vectors (HOG + LBP + GLCM) with {cpu_count()} CPU cores...")
t0 = time.time()
with Pool(processes=cpu_count()) as pool:
    X_train_raw = np.array(pool.map(extract_single_file_features, files_train), dtype=np.float32)
    X_val_raw   = np.array(pool.map(extract_single_file_features, files_val), dtype=np.float32)
    X_test_raw  = np.array(pool.map(extract_single_file_features, files_test), dtype=np.float32)
print(f"Feature extraction completed in {time.time()-t0:.2f} seconds.")
print(f"Feature matrix dimensions: {X_train_raw.shape}")

y_train = np.array(labels_train)
y_val = np.array(labels_val)
y_test = np.array(labels_test)

# Standardize feature vectors
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)

---
## Task 3: Model Training, Hyperparameter Tuning & Evaluation

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
import pandas as pd
import seaborn as sns

results = {}
probas = {}
cms = {}

# --- 1. Support Vector Classifier (SVM) ---
print("Training SVM with 5-Fold Cross-Validation...")
svm_grid = GridSearchCV(
    SVC(probability=True, random_state=42),
    {'C': [0.1, 1.0, 10.0], 'kernel': ['rbf'], 'gamma': ['scale']},
    cv=5, scoring='f1', n_jobs=-1
)
svm_grid.fit(X_train, y_train)
best_svm = svm_grid.best_estimator_

y_pred_svm = best_svm.predict(X_test)
y_proba_svm = best_svm.predict_proba(X_test)[:, 1]

results['SVM'] = {
    'Accuracy': accuracy_score(y_test, y_pred_svm),
    'Precision': precision_score(y_test, y_pred_svm),
    'Recall': recall_score(y_test, y_pred_svm),
    'F1-Score': f1_score(y_test, y_pred_svm),
    'AUROC': roc_auc_score(y_test, y_proba_svm)
}
probas['SVM'] = y_proba_svm
cms['SVM'] = confusion_matrix(y_test, y_pred_svm)
print(f"SVM Best Parameters: {svm_grid.best_params_}")

In [ ]:
# --- 2. Random Forest Classifier ---
print("Training Random Forest with 5-Fold Cross-Validation...")
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {'n_estimators': [50, 100], 'max_depth': [10, 20, None]},
    cv=5, scoring='f1', n_jobs=-1
)
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_

y_pred_rf = best_rf.predict(X_test)
y_proba_rf = best_rf.predict_proba(X_test)[:, 1]

results['Random Forest'] = {
    'Accuracy': accuracy_score(y_test, y_pred_rf),
    'Precision': precision_score(y_test, y_pred_rf),
    'Recall': recall_score(y_test, y_pred_rf),
    'F1-Score': f1_score(y_test, y_pred_rf),
    'AUROC': roc_auc_score(y_test, y_proba_rf)
}
probas['Random Forest'] = y_proba_rf
cms['Random Forest'] = confusion_matrix(y_test, y_pred_rf)
print(f"Random Forest Best Parameters: {rf_grid.best_params_}")

In [ ]:
# --- 3. Custom PyTorch CNN ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

torch.manual_seed(42)

class ChestXRayDataset(Dataset):
    def __init__(self, file_paths, labels):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transforms.ToTensor()
        
    def __len__(self):
        return len(self.file_paths)
        
    def __getitem__(self, idx):
        img = Image.open(self.file_paths[idx]).convert('L').resize((128, 128))
        return self.transform(img), self.labels[idx]

class PneumoniaCNN(nn.Module):
    def __init__(self):
        super(PneumoniaCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)
        
        self.fc1 = nn.Linear(128 * 16 * 16, 128)
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(128, 2)
        
    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

# DataLoaders
train_loader = DataLoader(ChestXRayDataset(files_train, labels_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(ChestXRayDataset(files_val, labels_val), batch_size=32, shuffle=False)
test_loader  = DataLoader(ChestXRayDataset(files_test, labels_test), batch_size=32, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f"Training PyTorch CNN on device: {device}")

cnn_model = PneumoniaCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001, weight_decay=1e-4)

epochs = 10
for epoch in range(epochs):
    cnn_model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outputs = cnn_model(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == lbls).sum().item()
        total += lbls.size(0)
    
    # Validation phase
    cnn_model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            val_correct += (torch.max(cnn_model(imgs), 1)[1] == lbls).sum().item()
            val_total += lbls.size(0)
    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Acc: {correct/total:.4f} | Val Acc: {val_correct/val_total:.4f}")

# Evaluate CNN on Test Set
cnn_model.eval()
test_preds, test_probas_cnn = [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(device)
        outputs = cnn_model(imgs)
        p = F.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        test_preds.extend(preds.cpu().numpy())
        test_probas_cnn.extend(p[:, 1].cpu().numpy())

test_preds = np.array(test_preds)
test_probas_cnn = np.array(test_probas_cnn)

results['Custom CNN'] = {
    'Accuracy': accuracy_score(y_test, test_preds),
    'Precision': precision_score(y_test, test_preds),
    'Recall': recall_score(y_test, test_preds),
    'F1-Score': f1_score(y_test, test_preds),
    'AUROC': roc_auc_score(y_test, test_probas_cnn)
}
probas['Custom CNN'] = test_probas_cnn
cms['Custom CNN'] = confusion_matrix(y_test, test_preds)

In [ ]:
# 4. Summary Table of Performance Metrics
df_metrics = pd.DataFrame(results).T.applymap(lambda x: f"{x:.4f}")
print("=== Comparative Performance Matrix ===")
display(df_metrics)

In [ ]:
# 5. Visualizations: Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for idx, (name, cm) in enumerate(cms.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Normal', 'Pneumonia'], yticklabels=['Normal', 'Pneumonia'])
    axes[idx].set_title(f"{name} Confusion Matrix", fontsize=12, fontweight='bold')
    axes[idx].set_xlabel("Predicted Label")
    axes[idx].set_ylabel("True Label")
plt.tight_layout()
plt.show()

In [ ]:
# 6. Visualizations: ROC Curves
plt.figure(figsize=(8, 6.5))
for name, p in probas.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    auc_val = results[name]['AUROC']
    plt.plot(fpr, tpr, label=f"{name} (AUROC = {auc_val:.4f})", linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', alpha=0.7)
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
plt.ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=11)
plt.title('Receiver Operating Characteristic (ROC) Curves Comparison', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Task 4: Clinical Discussion & Comparative Analysis

### 1. Clinical Impact of Misclassifications
- **False Negatives (Pneumonia classified as Normal):**  
  In clinical diagnostic workflows, **False Negatives represent the highest clinical risk**. A missed pneumonia diagnosis postpones antibiotic administration, potentially leading to acute respiratory distress syndrome (ARDS) or sepsis. False negatives occur primarily in early-stage consolidation or low-contrast opacity scans.
  
- **False Positives (Normal classified as Pneumonia):**  
  False positives lead to secondary diagnostic evaluations (e.g., CT scans or blood tests) and minor patient anxiety. While suboptimal, false positives are far preferable to false negatives in automated screening systems.

### 2. Feature Extraction vs. Deep Learning Feature Learning
- **Handcrafted Descriptors (HOG + LBP + GLCM) with SVM/RF:**  
  Traditional features capture fixed global edge orientations and micro-textures. However, they lack adaptive spatial translation invariance and cannot dynamically learn domain-specific pathological feature representations.
  
- **Custom PyTorch CNN (Top Performer):**  
  The CNN automatically learns hierarchical feature representations—from low-level edges in early conv layers to complex pathological lung consolidation patterns in deep layers. The PyTorch CNN achieves superior **Recall**, **F1-Score**, and **AUROC**, making it the recommended architecture for computer-aided diagnosis of pneumonia.